# Aula 8 — Padrões, outliers e formulação de hipóteses a partir dos dados
**Fund. de Programação, Dados e Estatística para IA** · Einstein Hospital Israelita

Objetivo: avançar da descrição de variáveis isoladas para uma investigação estruturada das relações na base.

> EDA gera pistas e hipóteses. Não prova causalidade nem substitui inferência estatística.

## 01 · Das variáveis às relações
**Onde a Aula 7 terminou:** um tempo de internação muito alto é apenas um valor extremo ou pertence a um perfil específico de paciente?

Um padrão tem várias explicações possíveis: diferença real entre populações, composição dos grupos, viés de seleção, qualidade dos dados, acaso e fatores não medidos. O papel da EDA é localizar a pergunta relevante.

### Leitura e verificação inicial
Toda análise começa confirmando qual arquivo foi carregado.

In [11]:
df = pd.read_csv("pacientes_aula8_analise.csv")
df.head()

,id_paciente,idade,faixa_etaria,servico,tempo_internacao_dias,internacoes_anteriores,readmissao_30d
0,P0001,60.0,60 ou mais,Cardiologia,3.0,2,0
1,P0002,39.0,18 a 59,Clínica Médica,6.0,0,0
2,P0003,83.0,60 ou mais,Clínica Médica,2.0,1,0
3,P0004,45.0,18 a 59,Oncologia,16.0,0,0
4,P0005,55.0,18 a 59,Clínica Médica,4.0,0,0


In [12]:
print(df.shape)
print(df.dtypes)
print(df.isna().sum())

(320, 7)
id_paciente                object
idade                     float64
faixa_etaria               object
servico                    object
tempo_internacao_dias     float64
internacoes_anteriores      int64
readmissao_30d              int64
dtype: object
id_paciente                0
idade                      6
faixa_etaria               6
servico                    0
tempo_internacao_dias     12
internacoes_anteriores     0
readmissao_30d             0
dtype: int64


## 02 · Cruzando categorias
### Serviço × readmissão
**Pergunta para a turma:** mais readmissões em número absoluto significam uma taxa maior?

In [13]:
tabela = pd.crosstab(df["servico"], df["readmissao_30d"])
tabela

readmissao_30d,0,1
servico,,
Cardiologia,71,2
Clínica Médica,81,13
Neurologia,37,7
Oncologia,44,7
Ortopedia,53,5


### De contagens para proporções
`normalize="index"` faz cada linha somar 100%. Uma proporção alta num grupo pequeno pede cautela.

In [14]:
tabela_pct = pd.crosstab(
    df["servico"],
    df["readmissao_30d"],
    normalize="index"
).mul(100).round(1)
tabela_pct

readmissao_30d,0,1
servico,,
Cardiologia,97.3,2.7
Clínica Médica,86.2,13.8
Neurologia,84.1,15.9
Oncologia,86.3,13.7
Ortopedia,91.4,8.6


### Margens mantêm o denominador visível

In [15]:
pd.crosstab(
    df["servico"],
    df["readmissao_30d"],
    margins=True
)

readmissao_30d,0,1,All
servico,,,
Cardiologia,71,2,73
Clínica Médica,81,13,94
Neurologia,37,7,44
Oncologia,44,7,51
Ortopedia,53,5,58
All,286,34,320


## 03 · Comparando grupos
### Tempo de internação por serviço
Compare média e mediana. Uma diferença entre médias não mostra que o serviço é a causa.

In [16]:
resumo_servico = df.groupby("servico")["tempo_internacao_dias"].agg(
    ["count", "mean", "median", "std", "min", "max"]
)
resumo_servico.round(2)

,count,mean,median,std,min,max
servico,,,,,,
Cardiologia,69,6.29,5.0,6.33,2.0,40.0
Clínica Médica,92,6.75,5.0,5.38,1.0,39.0
Neurologia,43,7.12,6.0,4.24,2.0,25.0
Oncologia,48,11.44,10.0,6.65,3.0,32.0
Ortopedia,56,3.30,3.0,1.17,2.0,6.0


### Pacientes com e sem readmissão
A resposta é descritiva: “na amostra observada, o grupo apresenta valores maiores”. Significância fica para a Aula 13.

In [17]:
df.groupby("readmissao_30d")[[
    "idade",
    "tempo_internacao_dias",
    "internacoes_anteriores"
]].agg(["count", "mean", "median", "std"]).round(2)

idade                      tempo_internacao_dias               \
               count   mean median    std                 count  mean median   
readmissao_30d                                                                 
0                280  58.89   59.0  16.45                   274  6.65    5.0   
1                 34  69.56   66.5  13.15                    34  8.00    6.0   

                     internacoes_anteriores                     
                 std                  count  mean median   std  
readmissao_30d                                                  
0               5.41                    286  0.99    1.0  1.03  
1               7.69                     34  1.59    1.0  1.50

## 04 · Composição dos grupos
### Agrupando por mais de uma variável
Grupos menores revelam estrutura escondida pela média global, mas tornam as estimativas instáveis.

In [18]:
df.groupby(["servico", "readmissao_30d"])["tempo_internacao_dias"].agg(
    ["count", "mean", "median"]
).round(2)

count   mean  median
servico        readmissao_30d                      
Cardiologia    0                  67   6.28     5.0
               1                   2   6.50     6.5
Clínica Médica 0                  79   6.58     6.0
               1                  13   7.77     5.0
Neurologia     0                  36   6.72     5.0
               1                   7   9.14     7.0
Oncologia      0                  41  11.46    11.0
               1                   7  11.29     8.0
Ortopedia      0                  51   3.33     3.0
               1                   5   3.00     3.0

### Estratificar pode mudar a história
Um padrão agregado pode enfraquecer, desaparecer ou inverter. Estratifique guiado por uma pergunta plausível.

In [19]:
pd.crosstab(
    [df["servico"], df["faixa_etaria"]],
    df["readmissao_30d"],
    normalize="index"
).mul(100).round(1)

readmissao_30d                   0     1
servico        faixa_etaria             
Cardiologia    18 a 59       100.0   0.0
               60 ou mais     95.7   4.3
Clínica Médica 18 a 59        97.7   2.3
               60 ou mais     75.5  24.5
Neurologia     18 a 59        94.4   5.6
               60 ou mais     73.9  26.1
               menor de 18   100.0   0.0
Oncologia      18 a 59        88.0  12.0
               60 ou mais     84.0  16.0
Ortopedia      18 a 59        97.1   2.9
               60 ou mais     81.8  18.2
               menor de 18   100.0   0.0

## 05 · Valores extremos em contexto
### Comece pela ordenação
As outras colunas ajudam a julgar se o valor é plausível.

In [20]:
df.nlargest(5, "tempo_internacao_dias")[[
    "idade",
    "servico",
    "tempo_internacao_dias",
    "internacoes_anteriores",
    "readmissao_30d"
]]

,idade,servico,tempo_internacao_dias,internacoes_anteriores,readmissao_30d
275,73.0,Cardiologia,40.0,1,0
295,89.0,Clínica Médica,39.0,1,1
303,82.0,Cardiologia,38.0,0,0
141,79.0,Clínica Médica,35.0,0,0
12,90.0,Oncologia,32.0,1,1


In [21]:
df.nsmallest(5, "tempo_internacao_dias")[[
    "idade",
    "servico",
    "tempo_internacao_dias",
    "readmissao_30d"
]]

,idade,servico,tempo_internacao_dias,readmissao_30d
38,65.0,Clínica Médica,1.0,0
2,83.0,Clínica Médica,2.0,0
10,47.0,Ortopedia,2.0,0
11,62.0,Ortopedia,2.0,0
18,22.0,Ortopedia,2.0,0


### Regra do IQR

In [22]:
q1 = df["tempo_internacao_dias"].quantile(0.25)
q3 = df["tempo_internacao_dias"].quantile(0.75)
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr
print("Q1:", q1, "| Q3:", q3, "| IQR:", iqr)
print("Limites:", limite_inferior, limite_superior)

Q1: 4.0 | Q3: 8.0 | IQR: 4.0
Limites: -2.0 14.0


### Selecionando os valores sinalizados
A regra identifica valores distantes da região central da distribuição.

In [23]:
mask_outlier = (
    (df["tempo_internacao_dias"] < limite_inferior) |
    (df["tempo_internacao_dias"] > limite_superior)
)
outliers_tempo = df.loc[mask_outlier]
outliers_tempo

,id_paciente,idade,faixa_etaria,servico,tempo_internacao_dias,internacoes_anteriores,readmissao_30d
3,P0004,45.0,18 a 59,Oncologia,16.0,0,0
12,P0013,90.0,60 ou mais,Oncologia,32.0,1,1
16,P0017,83.0,60 ou mais,Cardiologia,22.0,1,0
28,P0029,55.0,18 a 59,Neurologia,15.0,0,1
30,P0031,64.0,60 ou mais,Oncologia,30.0,1,0
49,P0050,70.0,60 ou mais,Oncologia,23.0,0,0
61,P0062,52.0,18 a 59,Clínica Médica,17.0,1,0
121,P0122,62.0,60 ou mais,Neurologia,25.0,1,0
141,P0142,79.0,60 ou mais,Clínica Médica,35.0,0,0
147,P0148,52.0,18 a 59,Neurologia,15.0,1,0


> “Outlier estatístico” não é sinônimo de “dado inválido”.

### Sinalizar sem apagar
Preserva todas as observações, permite comparar resultados com e sem extremos, mantém a regra rastreável e mostra se os extremos se concentram em algum grupo.

In [24]:
df["outlier_tempo_iqr"] = mask_outlier
df["outlier_tempo_iqr"].value_counts()

,count
outlier_tempo_iqr,
False,301
True,19


### Quem são as observações sinalizadas
Concentram-se num serviço, numa faixa etária? Têm mais internações anteriores?

In [25]:
df.loc[df["outlier_tempo_iqr"], [
    "idade", "faixa_etaria", "servico",
    "tempo_internacao_dias",
    "internacoes_anteriores", "readmissao_30d"
]].sort_values("tempo_internacao_dias", ascending=False)

,idade,faixa_etaria,servico,tempo_internacao_dias,internacoes_anteriores,readmissao_30d
275,73.0,60 ou mais,Cardiologia,40.0,1,0
295,89.0,60 ou mais,Clínica Médica,39.0,1,1
303,82.0,60 ou mais,Cardiologia,38.0,0,0
141,79.0,60 ou mais,Clínica Médica,35.0,0,0
12,90.0,60 ou mais,Oncologia,32.0,1,1
30,64.0,60 ou mais,Oncologia,30.0,1,0
191,43.0,18 a 59,Oncologia,28.0,0,0
121,62.0,60 ou mais,Neurologia,25.0,1,0
49,70.0,60 ou mais,Oncologia,23.0,0,0
261,75.0,60 ou mais,Oncologia,22.0,3,0


### Comparando extremos e demais
O resultado sugere uma pergunta. Não mostra que o tempo extremo provoca readmissão.

In [26]:
df.groupby("outlier_tempo_iqr")[[
    "idade",
    "internacoes_anteriores",
    "readmissao_30d"
]].agg(["count", "mean", "median"]).round(2)

idade               internacoes_anteriores               \
                  count   mean median                  count  mean median   
outlier_tempo_iqr                                                           
False               295  59.60   60.0                    301  1.06    1.0   
True                 19  66.95   70.0                     19  0.84    1.0   

                  readmissao_30d               
                           count  mean median  
outlier_tempo_iqr                              
False                        301  0.10    0.0  
True                          19  0.16    0.0

### Os extremos mexem mais na média
O cálculo avalia sensibilidade. Não justifica exclusão.

In [27]:
sem_extremos = df.loc[~df["outlier_tempo_iqr"]]
tempo_todos = df["tempo_internacao_dias"]
tempo_sem = sem_extremos["tempo_internacao_dias"]
print("Média:", tempo_todos.mean(), "→", tempo_sem.mean())
print("Mediana:", tempo_todos.median(), "→", tempo_sem.median())

Média: 6.8019480519480515 → 5.650519031141869
Mediana: 5.0 → 5.0


### Outlier não é propriedade do paciente
20 dias pode ser raro na população geral e esperado num subgrupo de alta complexidade.

**Pergunta para a turma:** faz sentido aplicar o mesmo limite a todos os serviços?

In [28]:
df.groupby("servico")["tempo_internacao_dias"].quantile(
    [0.25, 0.50, 0.75]
).unstack()

,0.25,0.50,0.75
servico,,,
Cardiologia,4.0,5.0,6.00
Clínica Médica,4.0,5.0,7.25
Neurologia,5.0,6.0,8.50
Oncologia,7.0,10.0,13.25
Ortopedia,2.0,3.0,4.00


## 06 · Do padrão à hipótese
Observação → Pergunta → Hipótese preliminar → Alternativas → Evidência necessária

**Preferir:** “Na amostra observada, o serviço A apresentou maior proporção de readmissão.” “Esse padrão sugere uma hipótese a investigar.”

**Evitar:** “O serviço A causa mais readmissões.” “Os outliers são erros e devem ser removidos.”

### Registro de achados
Preencha um bloco por achado.

- **Achado:**
- **Evidência:**
- **Hipótese:**
- **Limitação:**
- **Próximo passo:**

### Antes de concluir
Verifique grupos, distribuição, extremos e composição.

**Próxima aula — Aula 9:** escolhendo a visualização certa.